# Business Understanding

## Background

* The AIC Kijabe hospital's outpatient department (OPD) currently manages patient flow across approximately forty departments (General OPD, Casualty, Renal, Oncology, and others) without a structured way to anticipate periods of high patient volume. This can lead to under-staffing during surges and inefficient resource allocation during quieter periods.

## Business Objectives

* Enable the hospital to anticipate periods of higher patient volume in advance, so that staffing and resources can be planned proactively rather than reactively.
* Understand whether seasonal patterns (i.e rainy vs dry seasons in Kenya) affect patient volume, to support longer-term capacity planning.
* Understand patient return behavior (how long patients typically go before returning to the hospital) to support follow-up care planning and identify departments with outlying short or long return intervals.

## Data Analysis Goals

* Build a time series forecasting model to predict daily department-level patient arrival volume.
* Quantify the relationship between seasonal patterns and patient volume, including any lagged effects.
* Apply survival analysis to model time-to-return-visit, since discharge/exit data is not available in this dataset, return-visit interval is used as a signal for patient care continuity.

## Success Criteria

* Surge model - Forecast accuracy to be more useful than a simple "same as last week" baseline (e.g., a measurable improvement in MAE/RMSE).
* Seasonal correlation - A clear, evidence-based statement on whether seasons have a meaningful effect on patient volume.
* Survival analysis - Identifiable differences in return-visit patterns across at least one patient segment (e.g., department, age group).

# Data Preparation

In [4]:
# %pip install openpyxl

In [5]:
# import the necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

In [6]:
# load the data
data = pd.read_excel("Opd_data.xlsx")
data.head()

,PatientNumber,RegistrationDate,Gender,Age,QueuedTo,ConsultDescription
0,010575758,2024-06-01 00:00:00.000,Male,38 Yr(s),GENERAL OPD,GEneral Outpatient Care( NEW )
1,010575713,2024-06-01 00:00:00.000,Female,29 Yr(s),NaN,NaN
2,010569291,2024-06-01 00:04:38.083,Female,44 Yr(s),GENERAL OPD,General Outpatient Care
3,010575732,2024-06-01 00:04:42.460,Female,78 Yr(s),GENERAL OPD,General Outpatient Care
4,010575731,2024-06-01 00:10:16.037,Female,2 Yr(s),ADMISSION,Admission


In [7]:
# Basic exploratory
data.shape
print(data.info())

<class 'pandas.DataFrame'>
RangeIndex: 306306 entries, 0 to 306305
Data columns (total 6 columns):
 #   Column              Non-Null Count   Dtype         
---  ------              --------------   -----         
 0   PatientNumber       306306 non-null  str           
 1   RegistrationDate    306306 non-null  datetime64[us]
 2   Gender              306291 non-null  str           
 3   Age                 305735 non-null  str           
 4   QueuedTo            299528 non-null  str           
 5   ConsultDescription  299920 non-null  str           
dtypes: datetime64[us](1), str(5)
memory usage: 14.0 MB
None


## Data Cleaning

In [8]:
# Check for missing values
data.isnull().sum()

PatientNumber            0
RegistrationDate         0
Gender                  15
Age                    571
QueuedTo              6778
ConsultDescription    6386
dtype: int64

In [9]:
# check for duplicates
data.duplicated().sum()

np.int64(355)

In [10]:
# Drop the duplicates
data= data.drop_duplicates()

In [11]:
# Recheck for duplicates
data.duplicated().sum()

np.int64(0)

In [12]:
# Drop gender null values
data = data[data['Gender'].notna()]

In [13]:
# Standardize casing
data['Gender'] = data['Gender'].str.strip().str.upper()

## Age In Years

In [14]:
# extract the number and the unit separately
data['AgeNum'] = data['Age'].str.extract(r'(\d+)').astype(float)
# the numeric part, 38
data['AgeUnit'] = data['Age'].str.extract(r'\d+\s*(\D+)')[0].str.strip()  
# the unit part, "Yr(s)"

# convert based on unit: years stay the same, months divide by 12, weeks divide by 52, days divide by 365
conditions = [
    data['AgeUnit'] == 'Yr(s)',
    data['AgeUnit'] == 'Mth(s)',
    data['AgeUnit'] == 'Week(s)',
    data['AgeUnit'] == 'Days(s)'
]
choices = [
    data['AgeNum'],
    data['AgeNum'] / 12,
    data['AgeNum'] / 52,
    data['AgeNum'] / 365
]
data['AgeYears'] = np.select(conditions, choices, default=np.nan)

# round to 2 decimal places
data['AgeYears'] = data['AgeYears'].round(2)  

# only interested in age years
data = data.drop(columns=['AgeNum', 'AgeUnit'])


In [15]:
# Turning all impossible ages (>100)into NAN
data.loc[data['AgeYears'] > 100, 'AgeYears'] = np.nan

In [16]:
# calculate the mean age from all remaining valid values (this naturally excludes 
# both the >100 values we just nulled and any age that was already missing)
mean_age = round(data['AgeYears'].mean(), 2)
print(round(mean_age, 2))

40.86


In [17]:
# impute missing ages with mean
data['AgeYears'] = data['AgeYears'].fillna(mean_age)

print(mean_age)
# print(data['Age_num'].isna().sum())
print(data['AgeYears'].isna().sum())

40.86
0


In [18]:
# Bin into ages groups
data['AgeGroup']  = pd.cut(
    data['AgeYears'],                                    
    bins=[0,5,12,18,35,60,100],
    labels=['Infant(0-5)','Child(6-12)','Teen(13-18)','Young Adult(19-35)','Adult(36-60)','Senior(61-100)'],
    include_lowest=True
)

In [19]:
# Drop missing QueuedTo
data = data.dropna(subset=['QueuedTo'])

In [20]:
# Clean the QueuedTo column
data['QueuedTo']= data['QueuedTo'].str.strip().str.upper()

In [21]:
# Drop the age column
data = data.drop(columns=['Age'])

In [22]:
# Rechecking the null values
data.isna().sum()

PatientNumber         0
RegistrationDate      0
Gender                0
QueuedTo              0
ConsultDescription    0
AgeYears              0
AgeGroup              0
dtype: int64

In [23]:
data[data['AgeGroup'].isna()]['AgeYears'].describe()

count    0.0
mean     NaN
std      NaN
min      NaN
25%      NaN
50%      NaN
75%      NaN
max      NaN
Name: AgeYears, dtype: float64

In [24]:
data['AgeGroup'].unique()

['Adult(36-60)', 'Senior(61-100)', 'Infant(0-5)', 'Young Adult(19-35)', 'Child(6-12)', 'Teen(13-18)']
Categories (6, str): ['Infant(0-5)' < 'Child(6-12)' < 'Teen(13-18)' < 'Young Adult(19-35)' < 'Adult(36-60)' < 'Senior(61-100)']

In [25]:
# Recheck everything
print("Shape:", data.shape)
print()
print("Missing values:")
print(data.isnull().sum())
print()
print("Duplicate rows:", data.duplicated().sum())
print()
print("Dtypes:")
print(data.dtypes)
print()
print("Age range check:", data['AgeYears'].min(), "-", data['AgeYears'].max())
## print("Age range check:", data['Age_num'].min(), "-", data['Age_num'].max())
print()
print("Columns:", list(data.columns))

Shape: (299162, 7)

Missing values:
PatientNumber         0
RegistrationDate      0
Gender                0
QueuedTo              0
ConsultDescription    0
AgeYears              0
AgeGroup              0
dtype: int64



Duplicate rows: 0

Dtypes:
PatientNumber                    str
RegistrationDate      datetime64[us]
Gender                           str
QueuedTo                         str
ConsultDescription               str
AgeYears                     float64
AgeGroup                    category
dtype: object

Age range check: 0.0 - 100.0

Columns: ['PatientNumber', 'RegistrationDate', 'Gender', 'QueuedTo', 'ConsultDescription', 'AgeYears', 'AgeGroup']


In [26]:
# Exploring the data
data['QueuedTo'].unique()

<StringArray>
[                        'GENERAL OPD',                           'ADMISSION',
                            'CASUALTY',                        'GEN SURG OPD',
                               'RENAL',                                 'MCH',
                       'PHYSIOTHERAPY',                             'DAYCASE',
                              'DENTAL',                   'SPECIALITY CLINIC',
                          'PSYCHOLOGY',                                'OHNS',
                            'ONCOLOGY',                      'PRIVATE CLINIC',
                          'EYE CLINIC',                          'PAEDS BKKH',
 'CHRONIC CARE CLINIC (DM/HTN/TB/CCC)',                           'ORTHO OPD',
                          'PALLIATIVE',                           'AUDIOLOGY',
                           'NUTRITION',                'OCCUPATIONAL THERAPY',
          'NEURO SURGERY CONSULTATION',                     'FAMILY MEDICINE',
                     'DIABETIC CLINIC'

# Data Exploration

In [27]:
# Exploring the data
data['QueuedTo'].unique()

<StringArray>
[                        'GENERAL OPD',                           'ADMISSION',
                            'CASUALTY',                        'GEN SURG OPD',
                               'RENAL',                                 'MCH',
                       'PHYSIOTHERAPY',                             'DAYCASE',
                              'DENTAL',                   'SPECIALITY CLINIC',
                          'PSYCHOLOGY',                                'OHNS',
                            'ONCOLOGY',                      'PRIVATE CLINIC',
                          'EYE CLINIC',                          'PAEDS BKKH',
 'CHRONIC CARE CLINIC (DM/HTN/TB/CCC)',                           'ORTHO OPD',
                          'PALLIATIVE',                           'AUDIOLOGY',
                           'NUTRITION',                'OCCUPATIONAL THERAPY',
          'NEURO SURGERY CONSULTATION',                     'FAMILY MEDICINE',
                     'DIABETIC CLINIC'

In [28]:
data['QueuedTo'].value_counts(dropna=False)

QueuedTo
GENERAL OPD                            85620
ADMISSION                              33876
MCH                                    29451
CHRONIC CARE CLINIC (DM/HTN/TB/CCC)    25365
SPECIALITY CLINIC                      22356
ONCOLOGY                               13606
MEB SPECIALITY CLINIC                  13523
RENAL                                  11103
PAEDS BKKH                              9731
DENTAL                                  9252
DAYCASE                                 7596
OHNS                                    7290
EYE CLINIC                              6026
PHYSIOTHERAPY                           5825
GEN SURG OPD                            5155
CASUALTY                                3460
PRIVATE CLINIC                          3075
AUDIOLOGY                               1379
PSYCHOLOGY                              1223
NEUROSURGERY                            1032
OCCUPATIONAL THERAPY                     815
NUTRITION                                752
T

In [29]:
# 1. Drop the 3 one-off data errors
data = data[~data['QueuedTo'].isin(['KJ000104/19', 'Naivasha Town Clinic', 'All'])]

# 2. Flag admissions separately (so you can exclude/include as needed per analysis)
data['IsAdmission'] = data['QueuedTo'] == 'ADMISSION'


In [30]:
# Feature engineering
data['DayOfWeek'] = data['RegistrationDate'].dt.day_name()
data['IsWeekend'] = data['RegistrationDate'].dt.dayofweek >= 5
data['Month'] = data['RegistrationDate'].dt.month_name()
data['Date'] = data['RegistrationDate'].dt.date
data['Year'] = data['RegistrationDate'].dt.year
data['Hour'] = data['RegistrationDate'].dt.hour

data[['RegistrationDate', 'DayOfWeek', 'IsWeekend', 'Month', 'Hour', 'Year', 'Date']].head()

,RegistrationDate,DayOfWeek,IsWeekend,Month,Hour,Year,Date
0,2024-06-01 00:00:00.000,Saturday,True,June,0,2024,2024-06-01
2,2024-06-01 00:04:38.083,Saturday,True,June,0,2024,2024-06-01
3,2024-06-01 00:04:42.460,Saturday,True,June,0,2024,2024-06-01
4,2024-06-01 00:10:16.037,Saturday,True,June,0,2024,2024-06-01
5,2024-06-01 01:22:53.050,Saturday,True,June,1,2024,2024-06-01


In [31]:
# confirm this is ok
print(data.shape)
print(data.isna().sum())

(299161, 14)
PatientNumber         0
RegistrationDate      0
Gender                0
QueuedTo              0
ConsultDescription    0
AgeYears              0
AgeGroup              0
IsAdmission           0
DayOfWeek             0
IsWeekend             0
Month                 0
Date                  0
Year                  0
Hour                  0
dtype: int64


In [32]:
# Top 5 deparrtments, everything else grouped others
Top_Departments = [
    'GENERAL OPD',
    'CHRONIC CARE CLINIC (DM/HTN/TB/CCC)',
    'MCH',
    'SPECIALITY CLINIC',
    'ONCOLOGY'
]

data['QueuedTo_Grouped'] = data['QueuedTo'].where(
    data['QueuedTo'].isin(Top_Departments),'OTHER_OPD'
)

data['QueuedTo_Grouped'].value_counts()

QueuedTo_Grouped
OTHER_OPD                              122763
GENERAL OPD                             85620
MCH                                     29451
CHRONIC CARE CLINIC (DM/HTN/TB/CCC)     25365
SPECIALITY CLINIC                       22356
ONCOLOGY                                13606
Name: count, dtype: int64

In [33]:
# Exclude admission completely
data_opd_only = data[~data['IsAdmission']]

In [34]:
data.info()

<class 'pandas.DataFrame'>
Index: 299161 entries, 0 to 306305
Data columns (total 15 columns):
 #   Column              Non-Null Count   Dtype         
---  ------              --------------   -----         
 0   PatientNumber       299161 non-null  str           
 1   RegistrationDate    299161 non-null  datetime64[us]
 2   Gender              299161 non-null  str           
 3   QueuedTo            299161 non-null  str           
 4   ConsultDescription  299161 non-null  str           
 5   AgeYears            299161 non-null  float64       
 6   AgeGroup            299161 non-null  category      
 7   IsAdmission         299161 non-null  bool          
 8   DayOfWeek           299161 non-null  str           
 9   IsWeekend           299161 non-null  bool          
 10  Month               299161 non-null  str           
 11  Date                299161 non-null  object        
 12  Year                299161 non-null  int32         
 13  Hour                299161 non-null  int32   